# 05 - U-Net from scratch: the reference

**GPU runtime.** A small U-Net trained from scratch on the same training
glaciers, scored on the same validation glacier (SI), so the frozen encoders
have something to be measured against.

Kept fair to the linear probes:

| | probes (03) | U-Net (this) |
|---|---|---|
| training data | Crane, DBE, JAC, Jorum | same |
| classes | stone / glacier / ocean, NA ignored | same |
| loss | class-balanced | same |
| checkpoint choice on SI | none | none (fixed length; SI curve logged only) |
| input | per-encoder settings | `caffe` (official CaFFe statistics) |

Scored at **patch level** (16 x 16 px majority, same as the probes; saved as
`probe_zones__unet-scratch-48k__caffe__tile256.csv` so `04_compare` lists it with the
encoders) and at **pixel level** (`unet_pixel__...csv`, its native resolution).

Published CaFFe baseline (U-Net + ASPP, 256 px, test glaciers, pixel level,
mean over 5 runs): zone IoU ocean 0.82 / glacier 0.74 / rock 0.68 / NA 0.93,
mean 0.79 **including NA** (~0.75 without). Those are test-glacier numbers and
get compared in notebook 07, not here.

In [1]:
# --- cell: mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [3]:
# --- cell: GPU + data (train and SI only; test glaciers are not loaded) ---
from pathlib import Path
import torch
from sartransfer.data import inventory as inv
from sartransfer.baseline import load_split_images

assert torch.cuda.is_available(), "switch to a GPU runtime"
print("gpu:", torch.cuda.get_device_properties(0).name)
DRIVE = Path("/content/drive/MyDrive")
RAW, RES = DRIVE / "sar-transfer" / "raw", DRIVE / "sar-transfer" / "results"
inv.unzip_caffe(inv.fetch_caffe_zip(RAW), Path("/content/caffe"))
DATA = inv.find_data_root(Path("/content/caffe"))
df = inv.scan(DATA, subdirs=("sar_images",))

train_imgs = load_split_images(df, DATA, "train", norm="caffe")
val_imgs = load_split_images(df, DATA, "val", norm="caffe")
n_valid = lambda ims: sum(int((im["y"] != 255).sum()) for im in ims)
print(f"train: {len(train_imgs)} images, {n_valid(train_imgs):,} labelled pixels")
print(f"val (SI): {len(val_imgs)} images, {n_valid(val_imgs):,} labelled pixels")
assert len(train_imgs) == 438 and len(val_imgs) == 121, "expected the real CaFFe train/SI split"

gpu: NVIDIA A100-SXM4-40GB
already present: /content/drive/MyDrive/sar-transfer/raw/data_raw.zip  (2.86 GB)
copying zip to local disk (attempt 1) ...
extracted to /content/caffe
train: 438 images, 2,395,646,787 labelled pixels
val (SI): 121 images, 399,413,298 labelled pixels


In [4]:
# --- cell: SETTINGS + TRAIN (fixed length, 19.9 min on an A100-40GB) ---
from sartransfer.baseline import train_unet

# 48k fixed BEFORE seeing its result (2026-09-24): the 12k run was still improving.
# Earlier run kept on Drive as "unet-scratch__caffe__tile256" (12k iterations).
ITERS, BATCH, CROP, BASE = 48000, 16, 256, 32
RUN = f"unet-scratch-{ITERS // 1000}k__caffe__tile256"

model, log = train_unet(train_imgs, val_imgs, iters=ITERS, batch=BATCH, crop=CROP,
                        base=BASE, log_every=12000)   # SI check is info only
print(f"parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f} M")

class weights: {'stone': np.float32(2.759), 'glacier': np.float32(0.519), 'ocean': np.float32(1.409)}
  iter  12000  loss 0.4664  SI pixel mIoU 0.5199  6.0 min
  iter  24000  loss 0.2553  SI pixel mIoU 0.5840  10.6 min
  iter  36000  loss 0.1916  SI pixel mIoU 0.5792  15.3 min
  iter  48000  loss 0.1590  SI pixel mIoU 0.5851  19.9 min
parameters: 7.8 M


In [5]:
# --- cell: score the FINAL model on SI, patch + pixel level, save to Drive ---
import pandas as pd
from sartransfer.baseline import evaluate, to_table
from sartransfer.probe import iou_from_confusion

conf = evaluate(model, val_imgs, "cuda")
patch = to_table(conf["patch"], RUN)
pixel = to_table(conf["pixel"], RUN)
assert int(conf["pixel"]["band"]["all"].sum()) == n_valid(val_imgs), "every labelled SI pixel scored once"

patch.to_csv(RES / f"probe_zones__{RUN}.csv", index=False)       # listed by 04_compare
pixel.to_csv(RES / f"unet_pixel__{RUN}.csv", index=False)
log.to_csv(RES / f"unet_log__{RUN}.csv", index=False)
torch.save(model.state_dict(), RES / f"unet__{RUN}.pt")

cols = ["group", "IoU_stone", "IoU_glacier", "IoU_ocean", "mIoU", "n_patches"]
for name, t in (("PATCH level (compare with the probes)", patch), ("PIXEL level", pixel)):
    print(f"\n=== {name} ===")
    for g in ("band", "sensor_res"):
        print(t[t.grouping == g][cols].round(3).to_string(index=False))
print("\nsaved:", RUN, "(patch CSV, pixel CSV, training log, weights)")


=== PATCH level (compare with the probes) ===
group  IoU_stone  IoU_glacier  IoU_ocean  mIoU  n_patches
  all      0.506        0.680      0.582 0.590    1582515
    C      0.484        0.719      0.573 0.592     478503
    L      0.347        0.573      0.377 0.432      42608
    X      0.525        0.666      0.593 0.595    1061404
      group  IoU_stone  IoU_glacier  IoU_ocean  mIoU  n_patches
        all      0.506        0.680      0.582 0.590    1582515
ENVISAT@20m      0.502        0.745      0.734 0.660     112172
    ERS@20m      0.451        0.739      0.609 0.600     111473
 PALSAR@17m      0.347        0.573      0.377 0.432      42608
   RSAT@12m      0.496        0.695      0.484 0.558     229372
   RSAT@20m      0.448        0.738      0.481 0.556      25486
     TSX@7m      0.525        0.666      0.593 0.595    1061404

=== PIXEL level ===
group  IoU_stone  IoU_glacier  IoU_ocean  mIoU  n_patches
  all      0.496        0.679      0.580 0.585  399413298
    C      0.4

## Reading it

- **Patch-level mIoU vs the probes** (`04_compare`): C-RADIO 0.609, photo DINOv3
  0.584, satellite DINOv3 0.571; this U-Net 0.590.
  If the U-Net is clearly above them, frozen features leave accuracy on the
  table; if it is close or below, frozen photo features are competitive with a
  model trained on SAR from scratch.
- **Pixel vs patch level**: how much the 16 px patch grid itself costs.
- The SI curve in the training log is information only; the final model is
  scored, with no checkpoint picked on SI.